# Lab 7: Parquet and partitioning with Spark

Run the cells one at a time. The flights are in `../shared/output/flights`, the Parquet files are written to `../shared/output/parquet`, and the web UI of Spark is at <http://localhost:4040> while the session is open.

## Start Spark

In [ ]:
import os
import shutil
import time

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master('local[*]').appName('lab07').config(
    'spark.driver.memory', '4g').config('spark.sql.adaptive.enabled',
                                        'false').getOrCreate())
spark.sparkContext.defaultParallelism

## Read the flights

The same CSV files of lab 6, with the schema given explicitly. The helper `stats` counts the files of a folder and adds up their size.

In [ ]:
schema = ('flight_id long, flight_date date, airline_code string, '
          'flight_number string, tail_number string, origin string, '
          'destination string, scheduled_departure timestamp, '
          'actual_departure timestamp, distance_km int, cancelled int')
flights = spark.read.csv('../shared/output/flights', header=True, schema=schema)
OUT = '../shared/output/parquet'
shutil.rmtree(OUT, ignore_errors=True)


def stats(path, extension):
    files = [
        os.path.join(folder, name)
        for folder, _, names in os.walk(path)
        for name in names
        if name.endswith(extension)
    ]
    size = sum(os.path.getsize(f) for f in files) / 2**20
    return f'{len(files)} files, {size:.1f} MiB'


stats('../shared/output/flights', '.csv')

## Write the flights as Parquet

Spark writes a folder with one Parquet file for each partition of the flights. The default compression is snappy.

In [ ]:
start = time.time()
flights.write.parquet(f'{OUT}/flat')
print(f'{time.time() - start:.1f} s')
stats(f'{OUT}/flat', '.parquet')

## Write the flights partitioned by month

`partitionBy` creates one folder for each value of the column, named `column=value`. The column `month` is built from `flight_date`.

In [ ]:
by_month = flights.withColumn('month', F.date_format('flight_date', 'yyyy-MM'))
start = time.time()
by_month.write.partitionBy('month').parquet(f'{OUT}/by_month')
print(f'{time.time() - start:.1f} s')
print(sorted(f for f in os.listdir(f'{OUT}/by_month')
             if f.startswith('month=')))
stats(f'{OUT}/by_month', '.parquet')

## Partition pruning

The same question, the flights of March by airline, on both copies. On the partitioned copy the filter is on the partition column. On the flat copy it is on `flight_date`.

Look at `PartitionFilters` and `ReadSchema` in each plan.

In [ ]:
march_partitioned = (spark.read.parquet(f'{OUT}/by_month').filter(
    F.col('month') == '2026-03').groupBy('airline_code').count())
march_partitioned.explain()
march_partitioned.orderBy('airline_code').show()

In [ ]:
march_flat = (spark.read.parquet(f'{OUT}/flat').filter(
    F.date_format('flight_date', 'yyyy-MM') == '2026-03').groupBy(
        'airline_code').count())
march_flat.explain()
march_flat.orderBy('airline_code').show()

Open the SQL / DataFrame tab of the web UI and open the last two queries. The scan node reports the files and the bytes that each one read.

## Too many partitions

Partitioning by day makes one folder for each of the 273 days.

In [ ]:
by_day = flights.withColumn('day', F.col('flight_date').cast('string'))
start = time.time()
by_day.write.partitionBy('day').parquet(f'{OUT}/by_day')
print(f'{time.time() - start:.1f} s')
print(len([f for f in os.listdir(f'{OUT}/by_day') if f.startswith('day=')]),
      'folders')
stats(f'{OUT}/by_day', '.parquet')

## Compression

The same flights with three more codecs. Snappy is the default, so its size is the one of the flat copy.

In [ ]:
for codec in ['uncompressed', 'gzip', 'zstd']:
    start = time.time()
    flights.write.option('compression', codec).parquet(f'{OUT}/{codec}')
    print(f'{codec}: {time.time() - start:.1f} s,',
          stats(f'{OUT}/{codec}', '.parquet'))

## Stop Spark

In [ ]:
spark.stop()
spark